# Data Cleaning

In this notebook I am going to clean my data before doing any analysis.

**What I will do here step by step:**
1. Load the two CSV files
2. Merge them into one table
3. Check for missing values (nulls)
4. Check for duplicate rows
5. Fix the date column (it is stored as text, I need to convert it)
6. Rename confusing column names
7. Drop columns I don't need
8. Check unique values in text columns
9. Check for outliers
10. Save the cleaned data

> **Note:** I am NOT changing any values here. I am only fixing the structure and removing what is not useful.

# Step 1 - Importing Libraries and Loading the Data

In [1]:
# importing pandas to work with tables (dataframes)
import pandas as pd

# importing numpy for some math operations like percentile
import numpy as np

# importing os and Path to build file paths
import os
from pathlib import Path

# finding the project root folder
BASE_DIR = Path(os.getcwd())
if BASE_DIR.name == "notebooks":
    BASE_DIR = BASE_DIR.parent

# building the path to my raw data folder
RAW_DIR = BASE_DIR / "data" / "raw"

print("My raw data folder is:", RAW_DIR)

My raw data folder is: c:\Voyage_Project\data\raw


In [2]:
# loading the flights CSV file into a dataframe called flights
flights = pd.read_csv(RAW_DIR / "flights.csv")

# loading the users CSV file into a dataframe called users
users = pd.read_csv(RAW_DIR / "users.csv")

# printing the shape so I can see how many rows and columns each file has
print("flights shape:", flights.shape)
print("users shape  :", users.shape)

flights shape: (271888, 10)
users shape  : (1340, 5)


## Step 2 - Merge the Two Datasets

I need to combine flights and users into one table.

Both tables share a common column:
- In flights it is called `userCode`
- In users it is called `code`

I will rename `code` to `userCode` first, then merge.

In [3]:
# renaming the 'code' column in users to 'userCode'
# so it matches the column name in flights
users = users.rename(columns={"code": "userCode"})

# merging flights with users using the 'userCode' column
# how='left' means I keep ALL rows from flights
# even if a matching user is not found
df = flights.merge(users, on="userCode", how="left")

print("Merged dataframe shape:", df.shape)

# showing the first 3 rows to see the result
df.head(3)

Merged dataframe shape: (271888, 14)


,travelCode,userCode,from,to,flightType,price,time,distance,agency,date,company,name,gender,age
0,0,0,Recife (PE),Florianopolis (SC),firstClass,1434.38,1.76,676.53,FlyingDrops,09/26/2019,4You,Roy Braun,male,21
1,0,0,Florianopolis (SC),Recife (PE),firstClass,1292.29,1.76,676.53,FlyingDrops,09/30/2019,4You,Roy Braun,male,21
2,1,0,Brasilia (DF),Florianopolis (SC),firstClass,1487.52,1.66,637.56,CloudFy,10/03/2019,4You,Roy Braun,male,21


## Step 3 - Check for Missing Values (Nulls)

A null value means a cell is empty, the data is missing.

If I train a model on data with missing values it will give errors or bad results.
So I need to check this first.

In [4]:
# I am checking how many null values are in each column
# isnull() marks every empty cell as True
# sum() counts how many True values are in each column
null_counts = df.isnull().sum()

print("Null values in each column:")
print(null_counts)
print()

# I am also checking the total number of null values in the whole dataset
total_nulls = df.isnull().sum().sum()
print("Total null values in the entire dataset:", total_nulls)

Null values in each column:
travelCode    0
userCode      0
from          0
to            0
flightType    0
price         0
time          0
distance      0
agency        0
date          0
company       0
name          0
gender        0
age           0
dtype: int64

Total null values in the entire dataset: 0


**What I found:** No missing values at all every cell has data.
This is great news. I don't need to fill or drop any rows because of nulls.

## Step 4 - Check for Duplicate Rows

A duplicate row means the exact same booking appears more than once.
This would confuse the model, so I need to remove duplicates if any exist.

In [5]:
# I am counting how many rows are exact duplicates
# duplicated() marks duplicate rows as True
# sum() counts them
duplicate_count = df.duplicated().sum()

print("Number of duplicate rows:", duplicate_count)

Number of duplicate rows: 0


**What I found:** Zero duplicates. Nothing to remove here.

## Step 5 - Fix the Date Column

Right now the `date` column is stored as plain text (a string like "09/26/2019").
I need to convert it to a proper date format so I can extract things like month and year later.

This is called **type conversion** - changing a column from one data type to another.

In [6]:
# I am checking the current data type of the date column
print("Data type before fixing:", df["date"].dtype)

# I am showing what the date looks like right now (just text)
print("Sample date values before:", df["date"].head(3).tolist())

Data type before fixing: str
Sample date values before: ['09/26/2019', '09/30/2019', '10/03/2019']


In [7]:
# I am converting the date column from text to a proper datetime format
# format="%m/%d/%Y" tells pandas the text is in Month/Day/Year format
df["date"] = pd.to_datetime(df["date"], format="%m/%d/%Y")

# I am checking the data type again to confirm the conversion worked
print("Data type after fixing:", df["date"].dtype)

# I am showing what the date looks like now
print("Sample date values after:", df["date"].head(3).tolist())

Data type after fixing: datetime64[us]
Sample date values after: [Timestamp('2019-09-26 00:00:00'), Timestamp('2019-09-30 00:00:00'), Timestamp('2019-10-03 00:00:00')]


## Step 6 - Rename Confusing Column Names

The column `from` causes a problem in Python because `from` is a Python keyword
(used in `import pandas from ...`). This can cause confusion.

I will rename:
- `from`  →  `origin`  (where the flight starts)
- `to`    →  `destination`  (where the flight ends)

In [8]:
# renaming the columns to avoid the Python keyword problem
df = df.rename(columns={
    "from": "origin",       # 'from' is a Python keyword so I rename it
    "to"  : "destination"   # renaming 'to' as well for clarity
})

# printing all column names to confirm the rename worked
print("Columns after renaming:")
print(df.columns.tolist())

Columns after renaming:
['travelCode', 'userCode', 'origin', 'destination', 'flightType', 'price', 'time', 'distance', 'agency', 'date', 'company', 'name', 'gender', 'age']


## Step 7 - Drop Columns I Don't Need

Not every column is useful for my model or EDA.

I will drop:
- `travelCode` — this is just a trip ID number, it doesn't help predict price
- `name` — this is the person's full name, names don't predict flight prices

In [9]:
# I am dropping columns that are not useful
# 'travelCode' is just an ID number
# 'name' is the passenger's name - not useful for prediction
df = df.drop(columns=["travelCode", "name"])

# printing the remaining columns to confirm the drop worked
print("Remaining columns:", df.columns.tolist())
print()

# printing the new shape of the dataset
print("Shape after dropping columns:", df.shape)

Remaining columns: ['userCode', 'origin', 'destination', 'flightType', 'price', 'time', 'distance', 'agency', 'date', 'company', 'gender', 'age']

Shape after dropping columns: (271888, 12)


## Step 8 - Check Unique Values in Text Columns

My model only understands numbers, not text.
So I need to know what text values exist in each column
so I can convert them to numbers later (in notebook 06).

I will check: `flightType`, `agency`, `gender`, `company`

In [10]:
# I am checking unique values in the flightType column
print("flightType unique values:")
print(df["flightType"].value_counts())
print()

flightType unique values:
flightType
firstClass    116418
premium        78004
economic       77466
Name: count, dtype: int64



In [11]:
# I am checking unique values in the agency column
print("agency unique values:")
print(df["agency"].value_counts())
print()

agency unique values:
agency
Rainbow        116752
CloudFy        116378
FlyingDrops     38758
Name: count, dtype: int64



In [12]:
# I am checking unique values in the gender column
# Note: 'none' here means the user chose not to specify - it is NOT a missing value
print("gender unique values:")
print(df["gender"].value_counts())
print()

gender unique values:
gender
female    91580
male      91248
none      89060
Name: count, dtype: int64



In [13]:
# I am checking unique values in the company column
print("company unique values:")
print(df["company"].value_counts())
print()

company unique values:
company
4You             92986
Acme Factory     50944
Wonka Company    45882
Umbrella LTDA    41596
Monsters CYA     40480
Name: count, dtype: int64



**What I found and what I will do in notebook 06:**

| Column | Unique Values | What I will do |
|---|---|---|
| `flightType` | economic, premium, firstClass | Convert to numbers: 0, 1, 2 |
| `agency` | CloudFy, FlyingDrops, Rainbow | Convert to numbers: 0, 1, 2 |
| `gender` | male, female, none | Convert to numbers: 0, 1, 2 |
| `company` | 5 companies | Convert to numbers: 0, 1, 2, 3, 4 |

This process is called **Label Encoding** — giving each text category a number.

## Step 9 - Check for Outliers

An outlier is a value that is very far away from all other values.
For example, if most flight prices are between 300 and 1700 but one entry says 99999 — that is an outlier.

**How I will detect outliers by the IQR method:**
1. Find Q1 (the value at the 25% mark of the data)
2. Find Q3 (the value at the 75% mark of the data)
3. IQR = Q3 - Q1  (this is the middle 50% range)
4. Lower limit = Q1 - 1.5 × IQR
5. Upper limit = Q3 + 1.5 × IQR
6. Any value outside these limits is an outlier

I will check `price`, `time`, and `distance`.

In [14]:
# ── Outlier check for PRICE ──

# finding Q1 (value at 25th percentile)
Q1_price = df["price"].quantile(0.25)

# finding Q3 (value at 75th percentile)
Q3_price = df["price"].quantile(0.75)

# calculating IQR (the range between Q1 and Q3)
IQR_price = Q3_price - Q1_price

# calculating the lower and upper limits
lower_price = Q1_price - 1.5 * IQR_price
upper_price = Q3_price + 1.5 * IQR_price

# filtering rows where price is outside the limits
outliers_price = df[(df["price"] < lower_price) | (df["price"] > upper_price)]

print("=== PRICE ===")
print("Q1 (25th percentile):", round(Q1_price, 2))
print("Q3 (75th percentile):", round(Q3_price, 2))
print("IQR                 :", round(IQR_price, 2))
print("Lower limit         :", round(lower_price, 2))
print("Upper limit         :", round(upper_price, 2))
print("Number of outliers  :", len(outliers_price))

=== PRICE ===
Q1 (25th percentile): 672.66
Q3 (75th percentile): 1222.24
IQR                 : 549.58
Lower limit         : -151.71
Upper limit         : 2046.61
Number of outliers  : 0


In [15]:
# ── Outlier check for TIME (flight duration in hours) ──

Q1_time = df["time"].quantile(0.25)
Q3_time = df["time"].quantile(0.75)
IQR_time = Q3_time - Q1_time

lower_time = Q1_time - 1.5 * IQR_time
upper_time = Q3_time + 1.5 * IQR_time

outliers_time = df[(df["time"] < lower_time) | (df["time"] > upper_time)]

print("=== TIME ===")
print("Q1 (25th percentile):", round(Q1_time, 2))
print("Q3 (75th percentile):", round(Q3_time, 2))
print("IQR                 :", round(IQR_time, 2))
print("Lower limit         :", round(lower_time, 2))
print("Upper limit         :", round(upper_time, 2))
print("Number of outliers  :", len(outliers_time))

=== TIME ===
Q1 (25th percentile): 1.04
Q3 (75th percentile): 1.76
IQR                 : 0.72
Lower limit         : -0.04
Upper limit         : 2.84
Number of outliers  : 0


In [16]:
# ── Outlier check for DISTANCE (in km) ──

Q1_dist = df["distance"].quantile(0.25)
Q3_dist = df["distance"].quantile(0.75)
IQR_dist = Q3_dist - Q1_dist

lower_dist = Q1_dist - 1.5 * IQR_dist
upper_dist = Q3_dist + 1.5 * IQR_dist

outliers_dist = df[(df["distance"] < lower_dist) | (df["distance"] > upper_dist)]

print("=== DISTANCE ===")
print("Q1 (25th percentile):", round(Q1_dist, 2))
print("Q3 (75th percentile):", round(Q3_dist, 2))
print("IQR                 :", round(IQR_dist, 2))
print("Lower limit         :", round(lower_dist, 2))
print("Upper limit         :", round(upper_dist, 2))
print("Number of outliers  :", len(outliers_dist))

=== DISTANCE ===
Q1 (25th percentile): 401.66
Q3 (75th percentile): 676.53
IQR                 : 274.87
Lower limit         : -10.64
Upper limit         : 1088.84
Number of outliers  : 0


**What I found:** No outliers in any column.

This means all the values in price, time, and distance are within a normal range.
I don't need to remove or fix any rows because of outliers.

## Step 10 - Final Look at the Cleaned Data

In [17]:
# showing the final shape to see how many rows and columns I have
print("Final shape:", df.shape)
print()

# printing all column names and their data types
print("Column names and types:")
print(df.dtypes)
print()

# showing the first 5 rows of the cleaned dataset
print("First 5 rows:")
df.head()

Final shape: (271888, 12)

Column names and types:
userCode                int64
origin                    str
destination               str
flightType                str
price                 float64
time                  float64
distance              float64
agency                    str
date           datetime64[us]
company                   str
gender                    str
age                     int64
dtype: object

First 5 rows:


,userCode,origin,destination,flightType,price,time,distance,agency,date,company,gender,age
0,0,Recife (PE),Florianopolis (SC),firstClass,1434.38,1.76,676.53,FlyingDrops,2019-09-26,4You,male,21
1,0,Florianopolis (SC),Recife (PE),firstClass,1292.29,1.76,676.53,FlyingDrops,2019-09-30,4You,male,21
2,0,Brasilia (DF),Florianopolis (SC),firstClass,1487.52,1.66,637.56,CloudFy,2019-10-03,4You,male,21
3,0,Florianopolis (SC),Brasilia (DF),firstClass,1127.36,1.66,637.56,CloudFy,2019-10-04,4You,male,21
4,0,Aracaju (SE),Salvador (BH),firstClass,1684.05,2.16,830.86,CloudFy,2019-10-10,4You,male,21


In [18]:
# printing basic statistics for all numeric columns
# This shows me the min, max, mean etc of each column
df.describe().round(2)

,userCode,price,time,distance,date,age
count,271888.00,271888.00,271888.00,271888.00,271888,271888.00
mean,667.51,957.38,1.42,546.96,2021-01-11 05:25:56.052492,42.82
min,0.00,301.51,0.44,168.22,2019-09-26 00:00:00,21.00
25%,326.00,672.66,1.04,401.66,2020-04-02 00:00:00,32.00
50%,659.00,904.00,1.46,562.14,2020-11-14 00:00:00,42.00
75%,1011.00,1222.24,1.76,676.53,2021-09-09 00:00:00,54.00
max,1339.00,1754.17,2.44,937.77,2023-07-24 00:00:00,65.00
std,389.52,362.31,0.54,208.85,NaN,12.95


## Step 11 — Save the Cleaned Data

I will save the cleaned and merged dataframe as a new CSV file.
I am saving it to `data/processed/` folder so I always have the original raw files untouched.

In [19]:
# creating the processed folder if it does not already exist
PROCESSED_DIR = BASE_DIR / "data" / "processed"
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

# saving the cleaned dataframe as a new CSV file
# index=False means I don't want to save the row numbers as a column
df.to_csv(PROCESSED_DIR / "flights_merged.csv", index=False)

print("Cleaned data saved to:", PROCESSED_DIR / "flights_merged.csv")
print("Rows saved:", len(df))
print("Columns saved:", df.columns.tolist())

Cleaned data saved to: c:\Voyage_Project\data\processed\flights_merged.csv
Rows saved: 271888
Columns saved: ['userCode', 'origin', 'destination', 'flightType', 'price', 'time', 'distance', 'agency', 'date', 'company', 'gender', 'age']


## Summary - What I Did in This Notebook

| Step | What I did | Result |
|---|---|---|
| Load data | Loaded flights.csv and users.csv | 271,888 + 1,340 rows |
| Merge | Joined both on userCode | 271,888 rows, 13 columns |
| Null check | Checked for empty values | No nulls found |
| Duplicate check | Checked for repeated rows | No duplicates found |
| Fix date | Converted date from text to datetime | Done |
| Rename columns | from → origin, to → destination | Done |
| Drop columns | Removed travelCode and name | Done |
| Unique values | Checked text columns | 4 columns need encoding later |
| Outlier check | IQR method on price, time, distance | No outliers found |
| Save | Saved to data/processed/ | flights_merged.csv saved |
